In [213]:
#how does proximity to financial services, job density, population density, education level, and employment affect household income.

In [214]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
import scipy.stats as st
import statsmodels.api as sm 
import pylab as py 

# here are some of the tools we will use for our analyses
from sklearn.linear_model import LinearRegression
from sklearn.metrics import PredictionErrorDisplay
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score


import itertools
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor


from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm


import csv
import sqlite3
from pygam import LogisticGAM


In [215]:
income = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Household Income.csv", na_values="NA")
prox_bank = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Prox_bank_csv.csv", na_values="NA")
employment = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Employment.csv", na_values="NA")
job_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Job_Density_csv.csv", na_values="NA")
pop_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Population_Density_csv.csv", na_values="NA")
education = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Education Level - Bachelor's Degree.csv", na_values="NA")

In [216]:
income.columns

Index(['NPA', '2023'], dtype='object')

In [217]:
prox_bank.columns

Index(['NPA', '2023'], dtype='object')

In [218]:
employment.columns

Index(['NPA', '2023'], dtype='object')

In [219]:
job_density.columns

Index(['NPA', '2022'], dtype='object')

In [220]:
pop_density.columns

Index(['NPA', '2020'], dtype='object')

In [221]:
education.columns

Index(['NPA', '2023'], dtype='object')

In [222]:
conn = sqlite3.connect("my_data.db")

income.to_sql("income", conn, if_exists="replace", index=False)
prox_bank.to_sql("prox_bank", conn, if_exists="replace", index=False)
employment.to_sql("employment", conn, if_exists="replace", index=False)
job_density.to_sql("job_density", conn, if_exists="replace", index=False)
pop_density.to_sql("pop_density", conn, if_exists="replace", index=False)
education.to_sql("education", conn, if_exists="replace", index=False) 

459

In [223]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE income(
    NPA INT,
    median_income DOUBLE
)
""")

cursor.execute("""
CREATE TABLE prox_bank(
    NPA INT,
    "2023" DOUBLE
)
""")

cursor.execute("""
CREATE TABLE employment(
    NPA INT,
    employment_rate DOUBLE
)
""")

cursor.execute("""
CREATE TABLE job_density(
    NPA INT,
    "2022" DOUBLE
)
""")

cursor.execute("""
CREATE TABLE pop_density(
    NPA INT,
    "2020" DOUBLE
)
""")

cursor.execute("""
CREATE TABLE education(
    NPA INT,
    bachelors_percent DOUBLE
)
""")

In [224]:
for _, row in income.iterrows():
    cursor.execute("INSERT INTO income VALUES (?, ?)",
                   (row['NPA'], row['2023']))

for _, row in prox_bank.iterrows():
    cursor.execute("INSERT INTO prox_bank VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

for _, row in employment.iterrows():
    cursor.execute("INSERT INTO employment VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

for _, row in job_density.iterrows():
    cursor.execute("INSERT INTO job_density VALUES (?, ?)", 
                   (row['NPA'], row['2022']))

for _, row in pop_density.iterrows():
    cursor.execute("INSERT INTO pop_density VALUES (?, ?)", 
                   (row['NPA'], row['2020']))

for _, row in education.iterrows():
    cursor.execute("INSERT INTO education VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

In [225]:
def clean_npa(df):
    df['NPA'] = (
        df['NPA']
        .astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )
    return df

income = clean_npa(income)
prox_bank = clean_npa(prox_bank)
employment = clean_npa(employment)
job_density = clean_npa(job_density)
pop_density = clean_npa(pop_density)
education = clean_npa(education)

In [235]:
query = """
WITH all_npas AS (
    SELECT NPA FROM income
    UNION
    SELECT NPA FROM prox_bank
    UNION
    SELECT NPA FROM employment
    UNION
    SELECT NPA FROM job_density
    UNION
    SELECT NPA FROM pop_density
    UNION
    SELECT NPA FROM education
)

SELECT 
    a.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent
FROM all_npas a
LEFT JOIN income i ON a.NPA = i.NPA
LEFT JOIN prox_bank pb ON a.NPA = pb.NPA
LEFT JOIN employment e ON a.NPA = e.NPA
LEFT JOIN job_density jd ON a.NPA = jd.NPA
LEFT JOIN pop_density pd ON a.NPA = pd.NPA
LEFT JOIN education ed ON a.NPA = ed.NPA
"""

result = pd.read_sql_query(query, conn)
print(result.head())
print(result.info())

  NPA    income prox_bank employment job_density pop_density bachelors_percent
0   2   75084.0     0.245      0.956        None        None             0.352
1   3  117630.0       1.0      0.974        None        None             0.854
2   4  250001.0     0.152      0.943        None        None             0.894
3   5   49539.0      0.19      0.828        None        None             0.025
4   6   37907.0     0.699        1.0        None        None              0.21
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   NPA                918 non-null    object
 1   income             459 non-null    object
 2   prox_bank          459 non-null    object
 3   employment         459 non-null    object
 4   job_density        459 non-null    object
 5   pop_density        459 non-null    object
 6   bachelors_percent  459 non-null    object

In [226]:
query = """
SELECT 
    i.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent
FROM income i
LEFT JOIN prox_bank pb 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(pb.NPA AS TEXT))
LEFT JOIN employment e 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(e.NPA AS TEXT))
LEFT JOIN job_density jd 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(jd.NPA AS TEXT))
LEFT JOIN pop_density pd 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(pd.NPA AS TEXT))
LEFT JOIN education ed 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(ed.NPA AS TEXT))
"""

result = pd.read_sql_query(query, conn)
print(result.head())
print(result.info())

   NPA    income prox_bank employment job_density pop_density  \
0    2   75084.0     0.245      0.956        None        None   
1    3  117630.0       1.0      0.974        None        None   
2    4  250001.0     0.152      0.943        None        None   
3    5   49539.0      0.19      0.828        None        None   
4    6   37907.0     0.699        1.0        None        None   

  bachelors_percent  
0             0.352  
1             0.854  
2             0.894  
3             0.025  
4              0.21  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   NPA                459 non-null    int64 
 1   income             459 non-null    object
 2   prox_bank          459 non-null    object
 3   employment         459 non-null    object
 4   job_density        0 non-null      object
 5   pop_density        0 non-null      obje

In [227]:
query = """
WITH all_npas AS (
    SELECT NPA FROM income
    UNION
    SELECT NPA FROM prox_bank
    UNION
    SELECT NPA FROM employment
    UNION
    SELECT NPA FROM job_density
    UNION
    SELECT NPA FROM pop_density
    UNION
    SELECT NPA FROM education
)

SELECT 
    a.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent
FROM all_npas a
LEFT JOIN income i ON a.NPA = i.NPA
LEFT JOIN prox_bank pb ON a.NPA = pb.NPA
LEFT JOIN employment e ON a.NPA = e.NPA
LEFT JOIN job_density jd ON a.NPA = jd.NPA
LEFT JOIN pop_density pd ON a.NPA = pd.NPA
LEFT JOIN education ed ON a.NPA = ed.NPA
"""
result = pd.read_sql_query(query, conn)
print(result.head())
print(result.info())


  NPA    income prox_bank employment job_density pop_density bachelors_percent
0   2   75084.0     0.245      0.956        None        None             0.352
1   3  117630.0       1.0      0.974        None        None             0.854
2   4  250001.0     0.152      0.943        None        None             0.894
3   5   49539.0      0.19      0.828        None        None             0.025
4   6   37907.0     0.699        1.0        None        None              0.21
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   NPA                918 non-null    object
 1   income             459 non-null    object
 2   prox_bank          459 non-null    object
 3   employment         459 non-null    object
 4   job_density        459 non-null    object
 5   pop_density        459 non-null    object
 6   bachelors_percent  459 non-null    object

In [228]:
query = """
SELECT 
    i.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent
FROM income i
LEFT JOIN prox_bank pb ON TRIM(i.NPA) = TRIM(pb.NPA)
LEFT JOIN employment e ON TRIM(i.NPA) = TRIM(e.NPA)
LEFT JOIN job_density jd ON TRIM(i.NPA) = TRIM(jd.NPA)
LEFT JOIN pop_density pd ON TRIM(i.NPA) = TRIM(pd.NPA)
LEFT JOIN education ed ON TRIM(i.NPA) = TRIM(ed.NPA)

"""
result = pd.read_sql_query(query, conn)
print(result.head())
print(result.info())
print(result[['job_density', 'pop_density']].isna().sum())

   NPA    income prox_bank employment job_density pop_density  \
0    2   75084.0     0.245      0.956        None        None   
1    3  117630.0       1.0      0.974        None        None   
2    4  250001.0     0.152      0.943        None        None   
3    5   49539.0      0.19      0.828        None        None   
4    6   37907.0     0.699        1.0        None        None   

  bachelors_percent  
0             0.352  
1             0.854  
2             0.894  
3             0.025  
4              0.21  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   NPA                459 non-null    int64 
 1   income             459 non-null    object
 2   prox_bank          459 non-null    object
 3   employment         459 non-null    object
 4   job_density        0 non-null      object
 5   pop_density        0 non-null      obje

In [229]:
query = """
SELECT 
    i.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent
FROM income i
LEFT JOIN prox_bank pb ON i.NPA = pb.NPA
LEFT JOIN employment e ON i.NPA = e.NPA
LEFT JOIN job_density jd ON i.NPA = jd.NPA
LEFT JOIN pop_density pd ON i.NPA = pd.NPA
LEFT JOIN education ed ON i.NPA = ed.NPA

UNION

SELECT 
    pb.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent AS bachelors_percent
FROM prox_bank pb
LEFT JOIN income i ON pb.NPA = i.NPA
LEFT JOIN employment e ON pb.NPA = e.NPA
LEFT JOIN job_density jd ON pb.NPA = jd.NPA
LEFT JOIN pop_density pd ON pb.NPA = pd.NPA
LEFT JOIN education ed ON pb.NPA = ed.NPA

"""

result = pd.read_sql_query(query, conn)
print(result.head())
print(result.info())
print(result.tail())

   NPA    income prox_bank employment job_density pop_density  \
0    2   75084.0     0.245      0.956        None        None   
1    3  117630.0       1.0      0.974        None        None   
2    4  250001.0     0.152      0.943        None        None   
3    5   49539.0      0.19      0.828        None        None   
4    6   37907.0     0.699        1.0        None        None   

  bachelors_percent  
0             0.352  
1             0.854  
2             0.894  
3             0.025  
4              0.21  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   NPA                459 non-null    int64 
 1   income             459 non-null    object
 2   prox_bank          459 non-null    object
 3   employment         459 non-null    object
 4   job_density        0 non-null      object
 5   pop_density        0 non-null      obje

In [230]:
cursor.execute("SELECT COUNT(*) FROM job_density;")
result = cursor.fetchone()
print(result)

cursor.execute("SELECT COUNT(*) FROM pop_density;")
result = cursor.fetchone()
print(result)

(459,)
(459,)


In [231]:
cursor.execute("SELECT i.NPA FROM income i LEFT JOIN job_density jd ON i.NPA = jd.NPA WHERE jd.NPA IS NULL;")
result = cursor.fetchall()
print(result)

[(2,), (3,), (4,), (5,), (6,), (7,), (8,), (9,), (10,), (11,), (12,), (13,), (14,), (15,), (16,), (17,), (18,), (19,), (20,), (21,), (22,), (23,), (24,), (25,), (26,), (27,), (28,), (29,), (30,), (31,), (32,), (33,), (34,), (35,), (36,), (37,), (38,), (39,), (40,), (41,), (42,), (43,), (44,), (45,), (46,), (47,), (48,), (49,), (50,), (51,), (52,), (53,), (54,), (55,), (56,), (57,), (58,), (59,), (60,), (61,), (62,), (64,), (65,), (66,), (67,), (68,), (69,), (70,), (71,), (72,), (73,), (74,), (75,), (76,), (77,), (78,), (79,), (80,), (81,), (82,), (83,), (84,), (85,), (86,), (87,), (88,), (89,), (90,), (91,), (92,), (93,), (94,), (95,), (96,), (97,), (98,), (99,), (100,), (101,), (102,), (103,), (105,), (106,), (107,), (108,), (109,), (110,), (111,), (112,), (113,), (114,), (115,), (116,), (117,), (118,), (119,), (120,), (121,), (122,), (123,), (124,), (125,), (126,), (127,), (128,), (129,), (130,), (131,), (132,), (133,), (134,), (135,), (136,), (137,), (138,), (139,), (140,), (141,), 

In [232]:
print(pd.read_sql("PRAGMA table_info(job_density);", conn))
print(pd.read_sql("PRAGMA table_info(pop_density);", conn))

   cid  name    type  notnull dflt_value  pk
0    0   NPA     INT        0       None   0
1    1  2022  DOUBLE        0       None   0
   cid  name    type  notnull dflt_value  pk
0    0   NPA     INT        0       None   0
1    1  2020  DOUBLE        0       None   0


In [233]:
#Classification model starts here

In [234]:
df = result.copy()

# clean income
df['income'] = df['income'].replace('--', None)
df['income'] = df['income'].replace('[\$,]', '', regex=True)
df['income'] = pd.to_numeric(df['income'], errors='coerce')

# drop rows without income
df = df.dropna(subset=['income'])

# target
df['high_income'] = (df['income'] > df['income'].median()).astype(int)

<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:5: SyntaxWarning: invalid escape sequence '\$'
C:\Users\addis\AppData\Local\Temp\ipykernel_22516\561461919.py:5: SyntaxWarning: invalid escape sequence '\$'
  df['income'] = df['income'].replace('[\$,]', '', regex=True)


TypeError: list indices must be integers or slices, not str

In [ ]:
X = df[['prox_bank', 'employment', 'job_density', 'pop_density', 'bachelors_percent']]

# Target
y = df['high_income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=60
)

In [ ]:
print(type(result))

<class 'pandas.core.frame.DataFrame'>


In [ ]:
tree = DecisionTreeClassifier(max_depth=5)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))

NameError: name 'DecisionTreeClassifier' is not defined

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

y_pred_lda = lda.predict(X_test)

print("LDA Accuracy:", accuracy_score(y_test, y_pred_lda))

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print("kNN Accuracy:", accuracy_score(y_test, y_pred_knn))

In [ ]:
from pygam import LogisticGAM

gam = LogisticGAM()
gam.fit(X_train, y_train)

y_pred_gam = gam.predict(X_test)

print("GAM Accuracy:", accuracy_score(y_test, y_pred_gam))

In [ ]:
print("""
Model Comparison:
-----------------
Decision Tree:      {}
Logistic Regression: {}
LDA:                {}
kNN:                {}
GAM:                {}
""".format(
    accuracy_score(y_test, y_pred_tree),
    accuracy_score(y_test, y_pred_log),
    accuracy_score(y_test, y_pred_lda),
    accuracy_score(y_test, y_pred_knn),
    accuracy_score(y_test, y_pred_gam)
))